In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
from datetime import datetime, timedelta

In [ ]:
input_files = '/content/drive/MyDrive/airKorea_excel'

for root, dirs, files in os.walk(input_files):
    for file in files:
        if file.endswith(".xls"):
            file_path = os.path.join(root, file)

            df = pd.read_excel(file_path,header=3)

            filtered_df = df[df['측정망']=='도시대기']
            if "측정망" in filtered_df.columns:
                filtered_df = filtered_df.drop(columns=["측정망"])
            filtered_df=filtered_df[~df['측정소명'].str.contains('강화|백령')]

            columns = ["측정소명"]+[f'{i}시' for i in range(1,25)]
            filtered_df = filtered_df[columns]

            save_name=file.replace(".xls",".csv")
            save_path=os.path.join(root,save_name)

            filtered_df.to_csv(save_path, index=False, encoding="utf-8-sig")

/tmp/ipykernel_423/3153149102.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df=filtered_df[~df['측정소명'].str.contains('강화|백령')]
/tmp/ipykernel_423/3153149102.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df=filtered_df[~df['측정소명'].str.contains('강화|백령')]
/tmp/ipykernel_423/3153149102.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df=filtered_df[~df['측정소명'].str.contains('강화|백령')]
/tmp/ipykernel_423/3153149102.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df=filtered_df[~df['측정소명'].str.contains('강화|백령')]
/tmp/ipykernel_423/3153149102.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df=filtered_df[~df['측정소명'].str.contains('강화|백령')]
/tmp/ipykernel_423/3153149102.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df=fil

In [ ]:
dust_file = '/content/drive/MyDrive/airKorea_excel'

start_date = datetime(2026,1,1)
file_list = [] # pd.concat

for root, dirs, files in os.walk(dust_file):
    csv_file = sorted([file for file in files if file.endswith(".csv")])

    if not csv_file:
        continue

    for idx, file in enumerate(csv_file):
        input_file = os.path.join(root,file)
        df = pd.read_csv(input_file)
        #지역 생성 및 평균 계산
        df['측정소명'] = df['측정소명'].str.replace('[',' ').str.replace(']',' ')
        df['지역'] = df['측정소명'].str.split().str[0]
        hour_cols = [f'{i}시' for i in range(1,25)]
        df[hour_cols] = df[hour_cols].apply(pd.to_numeric,errors='coerce')
        columns_mean = df.groupby('지역')[hour_cols].mean().reset_index()

            # print(columns_mean)
            # wide -> long
        file_melt = columns_mean.melt(
            id_vars = '지역',
            value_vars=hour_cols,
            var_name='hour',
            value_name='PM10')

            # print(file_melt)

            # hour 시간 변환
        file_melt['hour'] = file_melt['hour'].str.replace('시','').astype(int)

            # 날짜 생성
        current_date = start_date + timedelta(days=idx)

        file_melt['datetime'] = (pd.to_datetime(current_date)+pd.to_timedelta(file_melt['hour']-1,unit='h'))

        file_melt = file_melt[['지역','datetime','PM10']]

        file_list.append(file_melt)

        pm_df = pd.concat(file_list,ignore_index=True)
        pm_df.to_csv("PM_processed.csv",index=False)

In [ ]:
# 기상 데이터와 미세 먼지의 공통 데이터 컬럼 맞추기. 지역과 시간.

df =pd.read_csv('/content/drive/MyDrive/OBS_ASOS_TIM_20260318081505.csv', encoding='cp949')

df=df.drop('지점',axis=1)
print(df.columns)
df = df.rename(columns={'지점명':'지역','일시':'datetime','기온(°C)':'temp','풍속(m/s)':'wind','습도(%)':'humidity'})

pm_df['datetime'] = pd.to_datetime(pm_df['datetime'])
df['datetime'] = pd.to_datetime(df['datetime'])


print(pm_df)
print(df)

df.to_csv('weather_processed.csv',encoding="utf-8-sig")

# 두 데이터 merge. 특정 열을 기준으로 정렬. 지역과 datetime 일치

merge_df = pd.merge(df, pm_df, on=['지역','datetime'],how='inner')
merge_df.to_csv("weather_pm.csv")

Index(['지점명', '일시', '기온(°C)', '풍속(m/s)', '습도(%)'], dtype='object')
       지역            datetime       PM10
0      부산 2026-01-01 00:00:00  11.923077
1      부산 2026-01-01 01:00:00  11.259259
2      부산 2026-01-01 02:00:00  11.296296
3      부산 2026-01-01 03:00:00  10.962963
4      부산 2026-01-01 04:00:00   8.888889
...    ..                 ...        ...
10795  대전 2026-03-16 19:00:00  28.818182
10796  대전 2026-03-16 20:00:00  20.272727
10797  대전 2026-03-16 21:00:00  17.545455
10798  대전 2026-03-16 22:00:00  14.363636
10799  대전 2026-03-16 23:00:00  16.545455

[10800 rows x 3 columns]
        지역            datetime  temp  wind  humidity
0      백령도 2026-01-01 01:00:00  -5.2   6.2      52.0
1      백령도 2026-01-01 02:00:00  -5.2   3.6      54.0
2      백령도 2026-01-01 03:00:00  -5.2   5.2      51.0
3      백령도 2026-01-01 04:00:00  -5.3   4.1      50.0
4      백령도 2026-01-01 05:00:00  -6.3   2.7      53.0
...    ...                 ...   ...   ...       ...
16186  북부산 2026-03-16 19:00:00  11.5   2.8  

In [ ]:
#Linear Regression 1. 미세먼지

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

data = pd.read_csv('/content/weather_pm.csv',usecols=['지역','datetime','PM10'])
data['datetime'] = pd.to_datetime(data['datetime'])

data = data.sort_values(['지역','datetime']).copy()

data['PM10_t-1'] = data.groupby('지역')['PM10'].shift(1)
data['PM10_t-2'] = data.groupby('지역')['PM10'].shift(2)

data['target_t+1'] = data.groupby('지역')['PM10'].shift(-1) #1시간 후
data['target_t+3'] = data.groupby('지역')['PM10'].shift(-3) #3시간 후

data = data.dropna()

In [ ]:
# 전처리
data = pd.get_dummies(data,columns=['지역'])

# 시계열 분리
train = data[data['datetime']<'2026-02-28'].copy()
test = data[data['datetime']>='2026-02-28'].copy()

# x, y 분리
x_train = train.drop(columns=['target_t+1','target_t+3','datetime'])
y_train_1 = train['target_t+1']
y_train_3 = train['target_t+3']

x_test = test.drop(columns=['target_t+1','target_t+3','datetime'])
y_test_1 = test['target_t+1']
y_test_3 = test['target_t+3']

In [ ]:
tscv = TimeSeriesSplit(n_splits=10)

# 과적합 판단 확인
train_r2_list1=[]
val_r2_list1=[]

train_r2_list3=[]
val_r2_list3=[]

mae_list1=[]
rmse_list1=[]
r2_score_list1=[]

mae_list3=[]
rmse_list3=[]
r2_score_list3=[]

for train_idx, val_idx in tscv.split(x_train):
    x_tr,x_val = x_train.iloc[train_idx],x_train.iloc[val_idx]
    y_tr1,y_val1 = y_train_1.iloc[train_idx],y_train_1.iloc[val_idx]
    y_tr3,y_val3 = y_train_3.iloc[train_idx],y_train_3.iloc[val_idx]

    model1 = LinearRegression()
    model1.fit(x_tr,y_tr1)

    pred_tr1=model1.predict(x_tr)
    pred_val1=model1.predict(x_val)

    train_r2_list1.append(r2_score(y_tr1,pred_tr1))
    val_r2_list1.append(r2_score(y_val1,pred_val1))

    pred = model1.predict(x_val)
    mae = mean_absolute_error(y_val1,pred)
    mae_list1.append(mae)

    rmse = root_mean_squared_error(y_val1,pred)
    rmse_list1.append(rmse)

    r2 = r2_score(y_val1,pred)
    r2_score_list1.append(r2)

    model3 = LinearRegression()
    model3.fit(x_tr,y_tr3)

    pred_tr3=model3.predict(x_tr)
    pred_val3=model3.predict(x_val)

    train_r2_list3.append(r2_score(y_tr3,pred_tr3))
    val_r2_list3.append(r2_score(y_val3,pred_val3))

    pred3 = model3.predict(x_val)

    mae3 = mean_absolute_error(y_val3,pred3)
    mae_list3.append(mae3)

    rmse3=root_mean_squared_error(y_val3,pred3)
    rmse_list3.append(rmse3)

    r2_3 = r2_score(y_val3,pred3)
    r2_score_list3.append(r2_3)

print("==== target+1 ====")
print("Train R2 mean : ",np.mean(train_r2_list1))
print("Val R2 mean : ",np.mean(val_r2_list1))
print("MAE_1 : ",np.mean(mae_list1))
print("RMSE_1: ",np.mean(rmse_list1))
print("R2 Score_1 : ",np.mean(r2_score_list1))

print("==== target+3 ====")
print("Train R2 mean : ",np.mean(train_r2_list3))
print("Val R2 mean : ",np.mean(val_r2_list3))
print("MAE_3 : ",np.mean(mae_list3))
print("RMSE_3 : ",np.mean(rmse_list3))
print("R2 Score_3 : ",np.mean(r2_score_list3))

==== target+1 ====
Train R2 mean :  0.9079608475873977
Val R2 mean :  0.9067444133710435
MAE_1 :  3.129662643613212
RMSE_1:  7.884174380413823
R2 Score_1 :  0.9067444133710435
==== target+3 ====
Train R2 mean :  0.46161057086172586
Val R2 mean :  0.4880643593153232
MAE_3 :  8.55504197343887
RMSE_3 :  19.151359580726297
R2 Score_3 :  0.4880643593153232


In [ ]:
# Linear Regression 미세먼지+기상
data = pd.read_csv('/content/weather_pm.csv')
data['datetime'] = pd.to_datetime(data['datetime'])

data = data.sort_values(['지역','datetime']).copy()

features = ['PM10','wind','humidity']

for col in features:
    data[f'{col}_t-1'] = data.groupby('지역')[col].shift(1)
    data[f'{col}_t-2'] = data.groupby('지역')[col].shift(2)

data['target_t+1']=data.groupby('지역')['PM10'].shift(-1)
data['target_t+3']=data.groupby('지역')['PM10'].shift(-3)

data = data.sort_values(['지역','datetime']).copy()

data = data.dropna()

In [ ]:
#one-hot encoding
data = pd.get_dummies(data,columns=['지역'])

#시계열 분리
train = data[data['datetime']<'2026-02-28'].copy()
test = data[data['datetime']>='2026-02-28'].copy()

#x,y 분리
x_train = train.drop(columns=['target_t+1','target_t+3','datetime'])
y_train_1=train['target_t+1']
y_train_3=train['target_t+3']

x_test = test.drop(columns=['target_t+1','target_t+3','datetime'])
y_test_1=test['target_t+1']
y_test_3=test['target_t+3']

In [ ]:
tscv = TimeSeriesSplit(n_splits=10)

# 과적합 판단 확인
train_r2_list1=[]
val_r2_list1=[]

train_r2_list3=[]
val_r2_list3=[]

mae_list1=[]
rmse_list1=[]
r2_score_list1=[]

mae_list3=[]
rmse_list3=[]
r2_score_list3=[]

for train_idx, val_idx in tscv.split(x_train):
    x_tr,x_val = x_train.iloc[train_idx],x_train.iloc[val_idx]
    y_tr1,y_val1 = y_train_1.iloc[train_idx],y_train_1.iloc[val_idx]
    y_tr3,y_val3 = y_train_3.iloc[train_idx],y_train_3.iloc[val_idx]

    model1 = LinearRegression()
    model1.fit(x_tr,y_tr1)

    pred_tr1=model1.predict(x_tr)
    pred_val1=model1.predict(x_val)

    train_r2_list1.append(r2_score(y_tr1,pred_tr1))
    val_r2_list1.append(r2_score(y_val1,pred_val1))

    pred = model1.predict(x_val)
    mae = mean_absolute_error(y_val1,pred)
    mae_list1.append(mae)

    rmse = root_mean_squared_error(y_val1,pred)
    rmse_list1.append(rmse)

    r2 = r2_score(y_val1,pred)
    r2_score_list1.append(r2)

    model3 = LinearRegression()
    model3.fit(x_tr,y_tr3)

    pred_tr3=model3.predict(x_tr)
    pred_val3=model3.predict(x_val)

    train_r2_list3.append(r2_score(y_tr3,pred_tr3))
    val_r2_list3.append(r2_score(y_val3,pred_val3))

    pred3 = model3.predict(x_val)

    mae3 = mean_absolute_error(y_val3,pred3)
    mae_list3.append(mae3)

    rmse3=root_mean_squared_error(y_val3,pred3)
    rmse_list3.append(rmse3)

    r2_3 = r2_score(y_val3,pred3)
    r2_score_list3.append(r2_3)

print("==== target+1 ====")
print("Train R2 mean : ",np.mean(train_r2_list1))
print("Val R2 mean : ",np.mean(val_r2_list1))
print("MAE_1 : ",np.mean(mae_list1))
print("RMSE_1: ",np.mean(rmse_list1))
print("R2 Score_1 : ",np.mean(r2_score_list1))

print("==== target+3 ====")
print("Train R2 mean : ",np.mean(train_r2_list3))
print("Val R2 mean : ",np.mean(val_r2_list3))
print("MAE_3 : ",np.mean(mae_list3))
print("RMSE_3 : ",np.mean(rmse_list3))
print("R2 Score_3 : ",np.mean(r2_score_list3))

==== target+1 ====
Train R2 mean :  0.9052640134864379
Val R2 mean :  0.8971855202668666
MAE_1 :  3.333724194962451
RMSE_1:  8.110169722404354
R2 Score_1 :  0.8971855202668666
==== target+3 ====
Train R2 mean :  0.46074596684626784
Val R2 mean :  0.4181396900062274
MAE_3 :  9.671461721662748
RMSE_3 :  20.232199009530703
R2 Score_3 :  0.4181396900062274


In [ ]:
#Random forest, 미세먼지만
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score

data = pd.read_csv('/content/weather_pm.csv',usecols=['지역','datetime','PM10'])

data['datetime'] = pd.to_datetime(data['datetime'])

data['PM10_t-1'] = data.groupby('지역')['PM10'].shift(1)
data['PM10_t-2'] = data.groupby('지역')['PM10'].shift(2)

data['target_t+1'] = data.groupby('지역')['PM10'].shift(-1) #1시간 후
data['target_t+3'] = data.groupby('지역')['PM10'].shift(-3) #3시간 후

data = data.dropna()

In [ ]:
#one-hot encoding
data = pd.get_dummies(data,columns=['지역'])

#시계열 분리
train = data[data['datetime']<'2026-02-28'].copy()
test = data[data['datetime']>='2026-02-28'].copy()

# x,y분리
x_train = train.drop(columns=['target_t+1','target_t+3','datetime'])
y_train_1=train['target_t+1']
y_train_3=train['target_t+3']

x_test = test.drop(columns=['target_t+1','target_t+3','datetime'])
y_test_1=test['target_t+1']
y_test_3=test['target_t+3']

In [ ]:
#target+1 best params by fold
# Fold 1: {'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': None}
# Fold 2: {'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': None}
# Fold 3: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 7: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# target+3 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# ==== target+1 ====
# MAE_1 :  3.333724194962451
# RMSE_1:  8.110169722404354
# R2 Score_1 :  0.8971855202668666
# Train R2: 0.9630573351553962
# Val R2  : 0.874207487180397
# Gap     : 0.08884984797499917
# ==== target+3 ====
# MAE_3 :  9.671461721662748
# RMSE_3 :  20.232199009530703
# R2 Score_3 :  0.4181396900062274
# Train R2: 0.7978514871480744
# Val R2  : 0.6067296690103438
# Gap     : 0.19112181813773066
# param_dist={
#     'n_estimators':[400,500,600],
#     'max_depth':[None,5,8,10,12],
#     'min_samples_split':[5,10,15],
#     'min_samples_leaf':[3,5,7,10],
#     'max_features':['sqrt','log2',None]
# }
# ================================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': None}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# target+3 best params by fold
# Fold 1: {'n_estimators': 50, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 4: {'n_estimators': 50, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 50, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 50, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 50, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# ==== target+1 ====
# MAE_1 :  3.333724194962451
# RMSE_1:  8.110169722404354
# R2 Score_1 :  0.8971855202668666
# Train R2: 0.9648646042221702
# Val R2  : 0.8809018855493964
# Gap     : 0.08396271867277383
# ==== target+3 ====
# MAE_3 :  9.671461721662748
# RMSE_3 :  20.232199009530703
# R2 Score_3 :  0.4181396900062274
# Train R2: 0.7949851480257026
# Val R2  : 0.6088616950256379
# Gap     : 0.18612345300006472
# best_params1 = {
#     'n_estimators': [400,500,600],
#     'min_samples_split': [5,10],
#     'min_samples_leaf': [3,5],
#     'max_features': [None],
#     'max_depth': [None,8]
#     }
# best_params3 = {
#     'n_estimators': [400,50],
#     'min_samples_split': [5,10],
#     'min_samples_leaf': [5,7],
#     'max_features': [None],
#     'max_depth': [8,10]
#     }
# ===================================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# target+3 best params by fold
# Fold 1: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# ==== target+1 ====
# MAE_1 :  3.333724194962451
# RMSE_1:  8.110169722404354
# R2 Score_1 :  0.8971855202668666
# Train R2: 0.9648620995915204
# Val R2  : 0.8808633883820193
# Gap     : 0.08399871120950109
# ==== target+3 ====
# MAE_3 :  9.671461721662748
# RMSE_3 :  20.232199009530703
# R2 Score_3 :  0.4181396900062274
# Train R2: 0.7921197017973801
# Val R2  : 0.6067196417580226
# Gap     : 0.1854000600393575
# best_params1 = {
#     'n_estimators': [500],
#     'min_samples_split': [5,10],
#     'min_samples_leaf': [3,5],
#     'max_features': [None],
#     'max_depth': [8]
#     }
# best_params3 = {
#     'n_estimators': [400,500],
#     'min_samples_split': [5,10],
#     'min_samples_leaf': [5,7],
#     'max_features': [None],
#     'max_depth': [8,10]
#     }
# =======================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# target+3 best params by fold
# Fold 1: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# ==== target+1 ====
# MAE_1 :  3.333724194962451
# RMSE_1:  8.110169722404354
# R2 Score_1 :  0.8971855202668666
# Train R2: 0.9678370852418927
# Val R2  : 0.8833983574847455
# Gap     : 0.08443872775714723
# ==== target+3 ====
# MAE_3 :  9.671461721662748
# RMSE_3 :  20.232199009530703
# R2 Score_3 :  0.4181396900062274
# Train R2: 0.7887817199309033
# Val R2  : 0.6055202154343291
# Gap     : 0.1832615044965742
# best_params1 = {
#     'n_estimators': [500],
#     'min_samples_split': [5],
#     'min_samples_leaf': [3],
#     'max_features': [None],
#     'max_depth': [8]
#     }
# best_params3 = {
#     'n_estimators': [400,500,600],
#     'min_samples_split': [5],
#     'min_samples_leaf': [7],
#     'max_features': [None],
#     'max_depth': [8,10]
#     }
# =================================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# target+3 best params by fold
# Fold 1: {'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# ==== target+1 ====
# MAE_1 :  3.333724194962451
# RMSE_1:  8.110169722404354
# R2 Score_1 :  0.8971855202668666
# Train R2: 0.9678370852418927
# Val R2  : 0.8833983574847455
# Gap     : 0.08443872775714723
# ==== target+3 ====
# MAE_3 :  9.671461721662748
# RMSE_3 :  20.232199009530703
# R2 Score_3 :  0.4181396900062274
# Train R2: 0.7908086604320416
# Val R2  : 0.6044997189365856
# Gap     : 0.18630894149545596
# best_params1 = {
#     'n_estimators': [500],
#     'min_samples_split': [5],
#     'min_samples_leaf': [3],
#     'max_features': [None],
#     'max_depth': [8]
#     }
# best_params3 = {
#     'n_estimators': [600,700],
#     'min_samples_split': [5,10],
#     'min_samples_leaf': [5,7],
#     'max_features': [None],
#     'max_depth': [8,10]
#     }


 | gap             | 해석            |
| --------------- | ------------- |
| **0 ~ 0.02**    | 거의 없음 (매우 좋음) |
| **0.02 ~ 0.05** | 약간 있음 (허용 가능) |
| **0.05 ~ 0.1**  | 약한 과적합        |
| **0.1 ~ 0.2**   | 과적합 있음        |
| **0.2 이상**      | 과적합 심함        |

이렇게 볼 수 있다.
target_t+1의 경우 과적합이 크지 않은 수준으로 나타났으나, target_t+3에서는 학습 데이터와 검증 데이터 간 성능 차이가 크게 나타나 과적합 경향이 확인되었다.
이에 따라, 과적합을 완화하기 위해 Random Forest의 하이퍼파라미터를 조정하고, 적절한 파라미터 조합을 탐색하고자 한다.

In [ ]:
outer_tscv = TimeSeriesSplit(n_splits=10)
inner_tscv = TimeSeriesSplit(n_splits=3)
rf = RandomForestRegressor(random_state=42,n_jobs=1)

# 과적합 판단 확인
train_r2_list1=[]
val_r2_list1=[]

train_r2_list3=[]
val_r2_list3=[]

#best params
best_params_1 = []
best_params_3 = []

best_score_1=[]
best_score_3=[]

mae_list_1 = []
rmse_list_1=[]
r2_score_list_1=[]

mae_list_3 = []
rmse_list_3=[]
r2_score_list_3=[]

best_params1 = {
    'n_estimators': [500],
    'min_samples_split': [5],
    'min_samples_leaf': [3],
    'max_features': [None],
    'max_depth': [8]
    }
best_params3 = {
    'n_estimators': [600,700],
    'min_samples_split': [5,10],
    'min_samples_leaf': [5,7],
    'max_features': [None],
    'max_depth': [8,10]
    }

for train_idx, val_idx in outer_tscv.split(x_train):
    x_tr,x_val = x_train.iloc[train_idx],x_train.iloc[val_idx]
    y_tr1,y_val1 = y_train_1.iloc[train_idx],y_train_1.iloc[val_idx]
    y_tr3,y_val3 = y_train_3.iloc[train_idx],y_train_3.iloc[val_idx]

    rf1 =RandomizedSearchCV(
        estimator=rf,
        param_distributions=best_params1,
        scoring='neg_mean_absolute_error',
        cv = inner_tscv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    rf3 = RandomizedSearchCV(
        estimator=rf,
        param_distributions=best_params3,
        n_iter=10,
        scoring='neg_mean_absolute_error',
        cv = inner_tscv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    rf1.fit(x_tr,y_tr1)
    best_model1 = rf1.best_estimator_

    best_params_1.append(rf1.best_params_)
    best_score_1.append(rf1.best_score_)

    pred_tr1=best_model1.predict(x_tr)
    pred_val1=best_model1.predict(x_val)

    train_r2_list1.append(r2_score(y_tr1,pred_tr1))
    val_r2_list1.append(r2_score(y_val1,pred_val1))

    mae1 = mean_absolute_error(y_val1,pred_val1)
    mae_list_1.append(mae1)

    rmse1 = root_mean_squared_error(y_val1,pred_val1)
    rmse_list_1.append(rmse1)

    r2_1 = r2_score(y_val1,pred_val1)
    r2_score_list_1.append(r2_1)

    rf3.fit(x_tr,y_tr3)
    best_model3 = rf3.best_estimator_

    best_params_3.append(rf3.best_params_)
    best_score_3.append(rf3.best_score_)

    pred_tr3=best_model3.predict(x_tr)
    pred_val3=best_model3.predict(x_val)

    train_r2_list3.append(r2_score(y_tr3,pred_tr3))
    val_r2_list3.append(r2_score(y_val3,pred_val3))

    mae3 = mean_absolute_error(y_val3,pred_val3)
    mae_list_3.append(mae3)

    rmse3 = root_mean_squared_error(y_val3,pred_val3)
    rmse_list_3.append(rmse3)

    r2_3 = r2_score(y_val3,pred_val3)
    r2_score_list_3.append(r2_3)

print("target+1 best params by fold")
for i, p in enumerate(best_params_1, 1):
    print(f"Fold {i}: {p}")

print("target+3 best params by fold")
for i, p in enumerate(best_params_3, 1):
    print(f"Fold {i}: {p}")

print("==== target+1 ====")
print("MAE_1 : ",np.mean(mae_list_1))
print("RMSE_1: ",np.mean(rmse_list_1))
print("R2 Score_1 : ",np.mean(r2_score_list_1))
print("Train R2:", np.mean(train_r2_list1))
print("Val R2  :", np.mean(val_r2_list1))
print("Gap     :", np.mean(train_r2_list1) - np.mean(val_r2_list1))

print("==== target+3 ====")
print("MAE_3 : ",np.mean(mae_list_3))
print("RMSE_3 : ",np.mean(rmse_list_3))
print("R2 Score_3 : ",np.mean(r2_score_list_3))
print("Train R2:", np.mean(train_r2_list3))
print("Val R2  :", np.mean(val_r2_list3))
print("Gap     :", np.mean(train_r2_list3) - np.mean(val_r2_list3))

Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 10 candidates, totalling 30 fits


...

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 10 candidates, totalling 30 fits
target+1 best params by fold
Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 2: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 3: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 5: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 8: {'n_estimators': 500, 'min_samples_spl

In [ ]:
print(len(mae_list_1), len(mae_list_3))
print(len(best_params_1), len(best_params_3))

10 10
10 10


In [ ]:
# Random forest 미세먼지 + 기상

data = pd.read_csv('/content/weather_pm.csv')
data['datetime'] = pd.to_datetime(data['datetime'])

data = data.sort_values(['지역','datetime']).copy()

features = ['PM10','wind','humidity']

for col in features:
    data[f'{col}_t-1'] = data.groupby('지역')[col].shift(1)
    data[f'{col}_t-2'] = data.groupby('지역')[col].shift(2)

data['target_t+1']=data.groupby('지역')['PM10'].shift(-1)
data['target_t+3']=data.groupby('지역')['PM10'].shift(-3)

data = data.sort_values(['지역','datetime']).copy()

data = data.dropna()

In [ ]:
#one-hot encoding
data = pd.get_dummies(data,columns=['지역'])

#시계열 분리
train = data[data['datetime']<'2026-02-28'].copy()
test = data[data['datetime']>='2026-02-28'].copy()

#x,y 분리
x_train = train.drop(columns=['target_t+1','target_t+3','datetime'])
y_train_1=train['target_t+1']
y_train_3=train['target_t+3']

x_test = test.drop(columns=['target_t+1','target_t+3','datetime'])
y_test_1=test['target_t+1']
y_test_3=test['target_t+3']

In [ ]:
# target+1 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': None}
# target+3 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 7: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 9: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# ==== target+1 ====
# MAE_1 :  3.0930320990742945
# RMSE_1:  9.062618974717235
# R2 Score_1 :  0.8743244831733964
# Train R2: 0.9281900055280483
# Val R2  : 0.8743244831733964
# Gap     : 0.05386552235465192
# ==== target+3 ====
# MAE_3 :  7.4059351535020355
# RMSE_3 :  16.883965476752227
# R2 Score_3 :  0.6042539188423018
# Train R2: 0.7141956063692396
# Val R2  : 0.6042539188423018
# Gap     : 0.10994168752693778
# param_dist={
#     'n_estimators':[400,500,600],
#     'max_depth':[None,5,8,10,12],
#     'min_samples_split':[5,10,15],
#     'min_samples_leaf':[3,5,7,10],
#     'max_features':['sqrt','log2',None]
# }
# =============================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 9: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# target+3 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 2: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 8}
# ==== target+1 ====
# MAE_1 :  3.0770161097178454
# RMSE_1:  9.200720659465981
# R2 Score_1 :  0.8707122195429611
# Train R2: 0.9201368721623856
# Val R2  : 0.8707122195429611
# Gap     : 0.049424652619424525
# ==== target+3 ====
# MAE_3 :  7.374804436193005
# RMSE_3 :  16.87401886415565
# R2 Score_3 :  0.6046037243793136
# Train R2: 0.7085982406995632
# Val R2  : 0.6046037243793136
# Gap     : 0.10399451632024959
# best_params1={
#     'n_estimators':[400,450,500],
#     'max_depth':[None,8,9,10],
#     'min_samples_split':[5,10],
#     'min_samples_leaf':[3,7],
#     'max_features':[None]
# }

# best_params3={
#     'n_estimators':[400,450,500],
#     'max_depth':[8,10],
#     'min_samples_split':[5,10],
#     'min_samples_leaf':[5,7],
#     'max_features':[None]
# }
# ================================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 2: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 3: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 4: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 5: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 6: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 7: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 8: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 9: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# Fold 10: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# target+3 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 4: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# ==== target+1 ====
# MAE_1 :  3.0180844481427123
# RMSE_1:  8.654751364799155
# R2 Score_1 :  0.8846237158664142
# Train R2: 0.9506202218322304
# Val R2  : 0.8846237158664142
# Gap     : 0.06599650596581619
# ==== target+3 ====
# MAE_3 :  7.345135226018013
# RMSE_3 :  16.732812806324027
# R2 Score_3 :  0.6116563581453061
# Train R2: 0.7088264214981832
# Val R2  : 0.6116563581453061
# Gap     : 0.09717006335287715
# best_params1={
#     'n_estimators':[400,450],
#     'max_depth':[9],
#     'min_samples_split':[5],
#     'min_samples_leaf':[3],
#     'max_features':[None]
# }

# best_params3={
#     'n_estimators':[400,500],
#     'max_depth':[10],
#     'min_samples_split':[10],
#     'min_samples_leaf':[7],
#     'max_features':[None]
# }
# ==============================================================================================================================
# target+1 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 2: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 3: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 6: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 8: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 9: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
# Fold 10: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 9}
# target+3 best params by fold
# Fold 1: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 2: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 3: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 4: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 5: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 6: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 7: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 8: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 9: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# Fold 10: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 10}
# ==== target+1 ====
# MAE_1 :  3.080837461673193
# RMSE_1:  9.20362739123156
# R2 Score_1 :  0.8706464476263065
# Train R2: 0.9197790511200077
# Val R2  : 0.8706464476263065
# Gap     : 0.04913260349370119
# ==== target+3 ====
# MAE_3 :  7.341011333213825
# RMSE_3 :  16.724693532391573
# R2 Score_3 :  0.6120269011025589
# Train R2: 0.7089204458123326
# Val R2  : 0.6120269011025589
# Gap     : 0.09689354470977374
# best_params1={
#     'n_estimators':[400,450,500],
#     'max_depth':[8,9],
#     'min_samples_split':[5],
#     'min_samples_leaf':[3,7],
#     'max_features':[None]
# }

# best_params3={
#     'n_estimators':[500],
#     'max_depth':[10],
#     'min_samples_split':[10],
#     'min_samples_leaf':[7],
#     'max_features':[None]
# }

In [ ]:
outer_tscv = TimeSeriesSplit(n_splits=10)
inner_tscv = TimeSeriesSplit(n_splits=3)
rf = RandomForestRegressor(random_state=42,n_jobs=1)

# 과적합 판단 확인
train_r2_list1=[]
val_r2_list1=[]

train_r2_list3=[]
val_r2_list3=[]

#best params
best_params_1 = []
best_params_3 = []

best_score_1=[]
best_score_3=[]

mae_list_1 = []
rmse_list_1=[]
r2_score_list_1=[]

mae_list_3 = []
rmse_list_3=[]
r2_score_list_3=[]

best_params1={
    'n_estimators':[400,450,500],
    'max_depth':[8,9],
    'min_samples_split':[5],
    'min_samples_leaf':[3,7],
    'max_features':[None]
}

best_params3={
    'n_estimators':[500],
    'max_depth':[10],
    'min_samples_split':[10],
    'min_samples_leaf':[7],
    'max_features':[None]
}

for train_idx, val_idx in outer_tscv.split(x_train):
    x_tr,x_val = x_train.iloc[train_idx],x_train.iloc[val_idx]
    y_tr1,y_val1 = y_train_1.iloc[train_idx],y_train_1.iloc[val_idx]
    y_tr3,y_val3 = y_train_3.iloc[train_idx],y_train_3.iloc[val_idx]

    rf1 =RandomizedSearchCV(
        estimator=rf,
        param_distributions=best_params1,
        n_iter=20,
        scoring='neg_mean_absolute_error',
        cv = inner_tscv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    rf3 = RandomizedSearchCV(
        estimator=rf,
        param_distributions=best_params3,
        n_iter=1,
        scoring='neg_mean_absolute_error',
        cv = inner_tscv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    rf1.fit(x_tr,y_tr1)
    best_model1 = rf1.best_estimator_

    best_params_1.append(rf1.best_params_)
    best_score_1.append(rf1.best_score_)

    pred_tr1=best_model1.predict(x_tr)
    pred_val1=best_model1.predict(x_val)

    train_r2_list1.append(r2_score(y_tr1,pred_tr1))
    val_r2_list1.append(r2_score(y_val1,pred_val1))

    mae1 = mean_absolute_error(y_val1,pred_val1)
    mae_list_1.append(mae1)

    rmse1 = root_mean_squared_error(y_val1,pred_val1)
    rmse_list_1.append(rmse1)

    r2_1 = r2_score(y_val1,pred_val1)
    r2_score_list_1.append(r2_1)

    rf3.fit(x_tr,y_tr3)
    best_model3 = rf3.best_estimator_

    best_params_3.append(rf3.best_params_)
    best_score_3.append(rf3.best_score_)

    pred_tr3=best_model3.predict(x_tr)
    pred_val3=best_model3.predict(x_val)

    train_r2_list3.append(r2_score(y_tr3,pred_tr3))
    val_r2_list3.append(r2_score(y_val3,pred_val3))

    mae3 = mean_absolute_error(y_val3,pred_val3)
    mae_list_3.append(mae3)

    rmse3 = root_mean_squared_error(y_val3,pred_val3)
    rmse_list_3.append(rmse3)

    r2_3 = r2_score(y_val3,pred_val3)
    r2_score_list_3.append(r2_3)

print("target+1 best params by fold")
for i, p in enumerate(best_params_1, 1):
    print(f"Fold {i}: {p}")

print("target+3 best params by fold")
for i, p in enumerate(best_params_3, 1):
    print(f"Fold {i}: {p}")

print("==== target+1 ====")
print("MAE_1 : ",np.mean(mae_list_1))
print("RMSE_1: ",np.mean(rmse_list_1))
print("R2 Score_1 : ",np.mean(r2_score_list_1))
print("Train R2:", np.mean(train_r2_list1))
print("Val R2  :", np.mean(val_r2_list1))
print("Gap     :", np.mean(train_r2_list1) - np.mean(val_r2_list1))

print("==== target+3 ====")
print("MAE_3 : ",np.mean(mae_list_3))
print("RMSE_3 : ",np.mean(rmse_list_3))
print("R2 Score_3 : ",np.mean(r2_score_list_3))
print("Train R2:", np.mean(train_r2_list3))
print("Val R2  :", np.mean(val_r2_list3))
print("Gap     :", np.mean(train_r2_list3) - np.mean(val_r2_list3))

Fitting 3 folds for each of 12 candidates, totalling 36 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


...

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
target+1 best params by fold
Fold 1: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
Fold 2: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
Fold 3: {'n_estimators': 450, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': None, 'max_depth': 8}
Fold 4: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 5: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 6: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 7: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 8}
Fold 8: {'n_estimators': 450, 'min_samples_spl

In [ ]:
print(len(mae_list_1), len(mae_list_3))
print(len(best_params_1), len(best_params_3))

10 10
10 10


inner = 파라미터 찾기

outer = 그 파라미터가 진짜 좋은지 평가

In [ ]:
# XGBoost. 미세먼지만.
# 기존 전처리 데이터 사용.
import xgboost as XGBRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

data = pd.read_csv('/content/weather_pm.csv',usecols=['지역','datetime','PM10'])

data['datetime'] = pd.to_datetime(data['datetime'])

data['PM10_t-1'] = data.groupby('지역')['PM10'].shift(1)
data['PM10_t-2'] = data.groupby('지역')['PM10'].shift(2)

data['target_t+1'] = data.groupby('지역')['PM10'].shift(-1) #1시간 후
data['target_t+3'] = data.groupby('지역')['PM10'].shift(-3) #3시간 후

data = data.dropna()

In [ ]:
#one-hot encoding
data = pd.get_dummies(data,columns=['지역'])

#시계열 분리
train = data[data['datetime']<'2026-02-28'].copy()
test = data[data['datetime']>='2026-02-28'].copy()

# x,y분리
x_train = train.drop(columns=['target_t+1','target_t+3','datetime'])
y_train_1=train['target_t+1']
y_train_3=train['target_t+3']

x_test = test.drop(columns=['target_t+1','target_t+3','datetime'])
y_test_1=test['target_t+1']
y_test_3=test['target_t+3']

In [ ]:
# target+1 best params by fold
# Fold 1: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 5, 'colsample_bytree': 0.7}
# Fold 8: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
# Fold 6: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
# Fold 8: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# ==== target+1 ====
# MAE_1 :  3.14256672609521
# RMSE_1:  9.690128412792664
# R2 Score_1 :  0.8658584197552674
# Train R2: 0.957537976679814
# Val R2  : 0.8658584197552674
# Gap     : 0.09167955692454655
# ==== target+3 ====
# MAE_3 :  7.441101189742352
# RMSE_3 :  17.130960364985512
# R2 Score_3 :  0.5776156890278713
# Train R2: 0.8556213460674877
# Val R2  : 0.5776156890278713
# Gap     : 0.27800565703961644
# param_dist = {
#     'n_estimators' : [200,300,400],
#     'learning_rate' : [0.01, 0.03, 0.1],
#     'max_depth' : [3,6,10],
#     'subsample' : [0.5, 0.7, 1.0],
#     'min_child_weight':[1,3,5,7],
#     'colsample_bytree' : [0.5,0.7,1.0],
#     'gamma' : [0,0.1,0.3,1,5]

# }
# ====================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.5, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# ==== target+1 ====
# MAE_1 :  3.2077872166584904
# RMSE_1:  9.68615205204852
# R2 Score_1 :  0.8666975595163591
# Train R2: 0.9535882189585175
# Val R2  : 0.8666975595163591
# Gap     : 0.08689065944215846
# ==== target+3 ====
# MAE_3 :  7.329899216399847
# RMSE_3 :  16.899710961220244
# R2 Score_3 :  0.5934335954567256
# Train R2: 0.8102066009304101
# Val R2  : 0.5934335954567256
# Gap     : 0.2167730054736845
# param_dist1 = {
#     'n_estimators' : [200,300],
#     'learning_rate' : [0.03,0.1],
#     'max_depth' : [3],
#     'subsample' : [0.7],
#     'min_child_weight':[3,5],
#     'colsample_bytree' : [0.7,1.0],
#     'gamma' : [0.1,0.3]
# }

# param_dist3 = {
#     'n_estimators' : [200,300],
#     'learning_rate' : [0.01,0.03,0.1],
#     'max_depth' : [3],
#     'subsample' : [0.5,0.7],
#     'min_child_weight':[5,7],
#     'colsample_bytree' : [0.7,1.0],
#     'gamma' : [0.3,1,5]
# }
# ====================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.5, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.5, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.5, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.5, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# ==== target+1 ====
# MAE_1 :  3.1591950817128653
# RMSE_1:  9.456740928033504
# R2 Score_1 :  0.8741309566942407
# Train R2: 0.9542121646615154
# Val R2  : 0.8741309566942407
# Gap     : 0.08008120796727469
# ==== target+3 ====
# MAE_3 :  7.2350276244275
# RMSE_3 :  16.719454564858335
# R2 Score_3 :  0.6022844679260823
# Train R2: 0.8138196942142842
# Val R2  : 0.6022844679260823
# Gap     : 0.21153522628820198
# param_dist1 = {
#     'n_estimators' : [200,300],
#     'learning_rate' : [0.1],
#     'max_depth' : [3],
#     'subsample' : [0.7],
#     'min_child_weight':[5],
#     'colsample_bytree' : [1.0],
#     'gamma' : [0.3]
# }

# param_dist3 = {
#     'n_estimators' : [200,300],
#     'learning_rate' : [0.03,0.1],
#     'max_depth' : [3],
#     'subsample' : [0.5,0.7],
#     'min_child_weight':[7],
#     'colsample_bytree' : [1.0],
#     'gamma' : [0.3,5]
# }
# =======================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 100, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 100, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.5, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 2, 'learning_rate': 0.01, 'gamma': 1, 'colsample_bytree': 0.7}
# Fold 2: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 9, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 3: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 4: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 5: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 6: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 9, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 7: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 8: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 9, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 10: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 9, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# ==== target+1 ====
# MAE_1 :  3.193783045745812
# RMSE_1:  9.456105449475752
# R2 Score_1 :  0.8734959890330195
# Train R2: 0.9443605780504789
# Val R2  : 0.8734959890330195
# Gap     : 0.0708645890174594
# ==== target+3 ====
# MAE_3 :  7.30644171123127
# RMSE_3 :  17.049975642707068
# R2 Score_3 :  0.5893823826636966
# Train R2: 0.7927197754346851
# Val R2  : 0.5893823826636966
# Gap     : 0.2033373927709885
# param_dist1 = {
#     'n_estimators' : [100,200,300],
#     'learning_rate' : [0.1],
#     'max_depth' : [3],
#     'subsample' : [0.7],
#     'min_child_weight':[5,7,10],
#     'colsample_bytree' : [1.0],
#     'gamma' : [0.3]
# }

# param_dist3 = {
#     'n_estimators': [100, 200, 300],
#     'learning_rate': [0.01, 0.03, 0.05],
#     'max_depth': [2, 3, 4],
#     'subsample': [0.5, 0.6, 0.7],
#     'min_child_weight': [5, 7, 9, 12],
#     'colsample_bytree': [0.5, 0.7, 0.9],
#     'gamma': [0.3, 1, 3, 5, 10],
#     'reg_alpha': [0, 0.1, 1],
#     'reg_lambda': [1, 3, 5]
# }
# =============================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 100, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 250, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 2: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 3: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 4: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 5: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 6: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 7: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 8: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 10: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 5, 'colsample_bytree': 0.9}
# ==== target+1 ====
# MAE_1 :  3.183934329674073
# RMSE_1:  9.406007842882456
# R2 Score_1 :  0.8753016977254846
# Train R2: 0.9440973193687789
# Val R2  : 0.8753016977254846
# Gap     : 0.06879562164329434
# ==== target+3 ====
# MAE_3 :  7.200731656114433
# RMSE_3 :  16.62763229506102
# R2 Score_3 :  0.609678460296721
# Train R2: 0.7889008811583629
# Val R2  : 0.609678460296721
# Gap     : 0.17922242086164186
# param_dist1 = {
#     'n_estimators' : [100,150,200,250,300],
#     'learning_rate' : [0.1],
#     'max_depth' : [3],
#     'subsample' : [0.7],
#     'min_child_weight':[5,7,10],
#     'colsample_bytree' : [1.0],
#     'gamma' : [0.3]
# }

# param_dist3 = {
#     'n_estimators': [250,300],
#     'learning_rate': [0.03, 0.05],
#     'max_depth': [2, 4],
#     'subsample': [0.6, 0.7],
#     'min_child_weight': [9, 12,14],
#     'colsample_bytree': [0.7,0.8, 0.9],
#     'gamma': [3, 5],
#     'reg_alpha': [0.1, 0.5,1],
#     'reg_lambda': [1]
# }
# ==============================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 250, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.9}
# Fold 2: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 3: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 4: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 350, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 5: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 350, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 6: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 7: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 350, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 8: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 350, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 10: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# ==== target+1 ====
# MAE_1 :  3.1804368730112875
# RMSE_1:  9.408358237885658
# R2 Score_1 :  0.8752368946838402
# Train R2: 0.9444208046133629
# Val R2  : 0.8752368946838402
# Gap     : 0.06918390992952261
# ==== target+3 ====
# MAE_3 :  7.183792604503867
# RMSE_3 :  16.576993120752352
# R2 Score_3 :  0.6110385871630954
# Train R2: 0.7887613100627837
# Val R2  : 0.6110385871630954
# Gap     : 0.17772272289968827
# param_dist1 = {
#     'n_estimators': [150, 200, 250],
#     'learning_rate': [0.1],
#     'max_depth': [3],
#     'subsample': [0.7],
#     'min_child_weight': [5, 7, 10],
#     'colsample_bytree': [1.0],
#     'gamma': [0.3]
# }

# param_dist3 = {
#     'n_estimators': [250, 300, 350],
#     'learning_rate': [0.03, 0.05],
#     'max_depth': [2, 3],
#     'subsample': [0.6, 0.7],
#     'min_child_weight': [12, 14, 16],
#     'colsample_bytree': [0.8, 0.9],
#     'gamma': [3, 5],
#     'reg_alpha': [0.5, 1],
#     'reg_lambda': [1]
# }
# ================================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 10: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 2: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 3: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 4: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 13, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 5: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 350, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 6: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 350, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 7: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 350, 'min_child_weight': 12, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 8: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 350, 'min_child_weight': 14, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 350, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# Fold 10: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 350, 'min_child_weight': 12, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.9}
# ==== target+1 ====
# MAE_1 :  3.1698157300660186
# RMSE_1:  9.41300887930646
# R2 Score_1 :  0.8749747791679676
# Train R2: 0.9481235869020065
# Val R2  : 0.8749747791679676
# Gap     : 0.07314880773403887
# ==== target+3 ====
# MAE_3 :  7.157536177680512
# RMSE_3 :  16.582460094276577
# R2 Score_3 :  0.6108298681199071
# Train R2: 0.7897327602601896
# Val R2  : 0.6108298681199071
# Gap     : 0.17890289214028243
# param_dist1 = {
#     'n_estimators': [150, 200, 250],
#     'learning_rate': [0.1],
#     'max_depth': [3],
#     'subsample': [0.7],
#     'min_child_weight': [5, 7],
#     'colsample_bytree': [1.0],
#     'gamma': [0.3]
# }

# param_dist3 = {
#     'n_estimators': [300, 350],
#     'learning_rate': [0.05],
#     'max_depth': [2, 3],
#     'subsample': [0.6],
#     'min_child_weight': [12, 13, 14],
#     'colsample_bytree': [0.9],
#     'gamma': [3],
#     'reg_alpha': [0.5, 1],
#     'reg_lambda': [1]
# }
# =====================================================

In [ ]:
# 하이퍼파라미터 튜닝

# 핵심 파라미터: n_estimators(트리 수), learning_rate(학습률), max_depth(트리 깊이), 회귀 ->objective : reg:squarederror

xgbmodel = XGBRegressor(
    objective='reg:squarederror', # 회귀 문제 설정
    random_state=42
)

# 과적합 판단 확인
train_r2_list1=[]
val_r2_list1=[]

train_r2_list3=[]
val_r2_list3=[]

#best params
best_params_1 = []
best_params_3 = []

best_score_1=[]
best_score_3=[]

mae_list_1 = []
rmse_list_1=[]
r2_score_list_1=[]

mae_list_3 = []
rmse_list_3=[]
r2_score_list_3=[]

param_dist1 = {
    'n_estimators': [150, 200, 250],
    'learning_rate': [0.1],
    'max_depth': [3],
    'subsample': [0.7],
    'min_child_weight': [5, 7, 10],
    'colsample_bytree': [1.0],
    'gamma': [0.3]
}

param_dist3 = {
    'n_estimators': [300, 350],
    'learning_rate': [0.05],
    'max_depth': [2, 3],
    'subsample': [0.6],
    'min_child_weight': [12, 13, 14],
    'colsample_bytree': [0.9],
    'gamma': [3],
    'reg_alpha': [0.5, 1],
    'reg_lambda': [1]
}

outer_tscv = TimeSeriesSplit(n_splits=10)
inner_tscv = TimeSeriesSplit(n_splits=3)

for train_idx, val_idx in outer_tscv.split(x_train):
    x_tr,x_val = x_train.iloc[train_idx],x_train.iloc[val_idx]
    y_tr1,y_val1 = y_train_1.iloc[train_idx],y_train_1.iloc[val_idx]
    y_tr3,y_val3 = y_train_3.iloc[train_idx],y_train_3.iloc[val_idx]

    random_search1 = RandomizedSearchCV(
        estimator=xgbmodel,
        param_distributions = param_dist1,
        n_iter = 20,
        scoring = 'neg_mean_squared_error',
        cv = inner_tscv,
        verbose = 1,
        n_jobs=-1,
        random_state=42
    )

    random_search3 = RandomizedSearchCV(
        estimator=xgbmodel,
        param_distributions = param_dist3,
        n_iter = 20,
        scoring = 'neg_mean_squared_error',
        cv = inner_tscv,
        verbose = 1,
        n_jobs=-1,
        random_state=42
    )

    random_search1.fit(x_tr,y_tr1)
    best_model1 = random_search1.best_estimator_

    best_params_1.append(random_search1.best_params_)
    best_score_1.append(random_search1.best_score_)

    pred_tr1=best_model1.predict(x_tr)
    pred_val1=best_model1.predict(x_val)

    train_r2_list1.append(r2_score(y_tr1,pred_tr1))
    val_r2_list1.append(r2_score(y_val1,pred_val1))

    mae1 = mean_absolute_error(y_val1,pred_val1)
    mae_list_1.append(mae1)

    rmse1 = root_mean_squared_error(y_val1,pred_val1)
    rmse_list_1.append(rmse1)

    r2_1 = r2_score(y_val1,pred_val1)
    r2_score_list_1.append(r2_1)

    random_search3.fit(x_tr,y_tr3)
    best_model3 = random_search3.best_estimator_

    best_params_3.append(random_search3.best_params_)
    best_score_3.append(random_search3.best_score_)

    pred_tr3=best_model3.predict(x_tr)
    pred_val3=best_model3.predict(x_val)

    train_r2_list3.append(r2_score(y_tr3,pred_tr3))
    val_r2_list3.append(r2_score(y_val3,pred_val3))

    mae3 = mean_absolute_error(y_val3,pred_val3)
    mae_list_3.append(mae3)

    rmse3 = root_mean_squared_error(y_val3,pred_val3)
    rmse_list_3.append(rmse3)

    r2_3 = r2_score(y_val3,pred_val3)
    r2_score_list_3.append(r2_3)

print("target+1 best params by fold")
for i, p in enumerate(best_params_1, 1):
    print(f"Fold {i}: {p}")

print("target+3 best params by fold")
for i, p in enumerate(best_params_3, 1):
    print(f"Fold {i}: {p}")

print("==== target+1 ====")
print("MAE_1 : ",np.mean(mae_list_1))
print("RMSE_1: ",np.mean(rmse_list_1))
print("R2 Score_1 : ",np.mean(r2_score_list_1))
print("Train R2:", np.mean(train_r2_list1))
print("Val R2  :", np.mean(val_r2_list1))
print("Gap     :", np.mean(train_r2_list1) - np.mean(val_r2_list1))

print("==== target+3 ====")
print("MAE_3 : ",np.mean(mae_list_3))
print("RMSE_3 : ",np.mean(rmse_list_3))
print("R2 Score_3 : ",np.mean(r2_score_list_3))
print("Train R2:", np.mean(train_r2_list3))
print("Val R2  :", np.mean(val_r2_list3))
print("Gap     :", np.mean(train_r2_list3) - np.mean(val_r2_list3))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


...

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
target+1 best params by fold
Fold 1: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
Fold 2: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
Fold 4: {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
Fold 5: {'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
Fold 6: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1,

randomSearch로 best_params를 찾고, 그 구간을 GridSearchCV로 좁게 탐색하는 편이 효율적.

In [ ]:
# XGBoost 미세먼지 + 기상데이터
data = pd.read_csv('/content/weather_pm.csv')
data['datetime'] = pd.to_datetime(data['datetime'])

data = data.sort_values(['지역','datetime']).copy()

features = ['PM10','wind','humidity']

for col in features:
    data[f'{col}_t-1'] = data.groupby('지역')[col].shift(1)
    data[f'{col}_t-2'] = data.groupby('지역')[col].shift(2)

data['target_t+1']=data.groupby('지역')['PM10'].shift(-1)
data['target_t+3']=data.groupby('지역')['PM10'].shift(-3)

data = data.sort_values(['지역','datetime']).copy()

data = data.dropna()

#one-hot encoding
data = pd.get_dummies(data,columns=['지역'])

#시계열 분리
train = data[data['datetime']<'2026-02-28'].copy()
test = data[data['datetime']>='2026-02-28'].copy()

#x,y 분리
x_train = train.drop(columns=['target_t+1','target_t+3','datetime'])
y_train_1=train['target_t+1']
y_train_3=train['target_t+3']

x_test = test.drop(columns=['target_t+1','target_t+3','datetime'])
y_test_1=test['target_t+1']
y_test_3=test['target_t+3']

In [ ]:
# target+1 best params by fold
# Fold 1: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.1, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
# Fold 10: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 1, 'colsample_bytree': 0.5}
# Fold 2: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 5, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
# Fold 7: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
# Fold 8: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 10, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 1.0}
# Fold 9: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.5}
# Fold 10: {'subsample': 0.5, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
# ==== target+1 ====
# MAE_1 :  3.64138266431936
# RMSE_1:  9.780239452189084
# R2 Score_1 :  0.8561137629773843
# Train R2: 0.9748097117315513
# Val R2  : 0.8561137629773843
# Gap     : 0.118695948754167
# ==== target+3 ====
# MAE_3 :  8.82279552615015
# RMSE_3 :  19.159261637505594
# R2 Score_3 :  0.41032006038204427
# Train R2: 0.855233984915419
# Val R2  : 0.41032006038204427
# Gap     : 0.44491392453337475
# param_dist = {
#     'n_estimators' : [200,300,400],
#     'learning_rate' : [0.01, 0.03, 0.1],
#     'max_depth' : [3,6,10],
#     'subsample' : [0.5, 0.7, 1.0],
#     'min_child_weight':[1,3,5,7],
#     'colsample_bytree' : [0.5,0.7,1.0],
#     'gamma' : [0,0.1,0.3,1,5]

# }
# ========================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 0.8}
# Fold 9: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.3, 'colsample_bytree': 0.8}
# Fold 10: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 2: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 3: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 4: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 5: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 6: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 7: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 8: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 10: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# ==== target+1 ====
# MAE_1 :  3.7847785350980807
# RMSE_1:  10.090558857630791
# R2 Score_1 :  0.8436507416745362
# Train R2: 0.9658624308434431
# Val R2  : 0.8436507416745362
# Gap     : 0.1222116891689069
# ==== target+3 ====
# MAE_3 :  7.735469741107101
# RMSE_3 :  17.190492617252225
# R2 Score_3 :  0.5867829908012799
# Train R2: 0.6487440505212982
# Val R2  : 0.5867829908012799
# Gap     : 0.06196105972001831
# param_dist1 = {
#     'n_estimators': [200, 300, 500],
#     'learning_rate': [0.05, 0.07, 0.1],
#     'max_depth': [3, 4],
#     'subsample': [0.8],
#     'min_child_weight': [5, 7, 10],
#     'colsample_bytree': [0.8, 1.0],
#     'gamma': [0.3, 1],
#     'reg_alpha': [0, 0.1],
#     'reg_lambda': [1, 3]
# }

# param_dist3 = {
#     'n_estimators': [200, 300, 700],
#     'learning_rate': [0.03, 0.05],
#     'max_depth': [2],
#     'subsample': [0.6],
#     'min_child_weight': [17],
#     'colsample_bytree': [0.8],
#     'gamma': [3, 5],
#     'reg_alpha': [0.1],
#     'reg_lambda': [1, 3]
# }
# ==========================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 0.3, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 7: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 0.8}
# Fold 9: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 0.8}
# Fold 10: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.07, 'gamma': 0.3, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 2: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 3: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 4: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 5: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 6: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 7: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 8: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 10: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# ==== target+1 ====
# MAE_1 :  3.7795735373705774
# RMSE_1:  9.920656938814705
# R2 Score_1 :  0.8422697820376047
# Train R2: 0.9817417920133366
# Val R2  : 0.8422697820376047
# Gap     : 0.13947200997573184
# ==== target+3 ====
# MAE_3 :  7.901013310443335
# RMSE_3 :  17.316110198300308
# R2 Score_3 :  0.5764585085718099
# Train R2: 0.6561480817322156
# Val R2  : 0.5764585085718099
# Gap     : 0.0796895731604057
# param_dist1 = {
#     'n_estimators': [200,500],
#     'learning_rate': [0.05, 0.07],
#     'max_depth': [3, 4],
#     'subsample': [0.8],
#     'min_child_weight': [5,10,15],
#     'colsample_bytree': [0.8, 1.0],
#     'gamma': [0.3, 1],
#     'reg_alpha': [0, 0.1],
#     'reg_lambda': [1, 3]
# }

# param_dist3 = {
#     'n_estimators': [200, 700],
#     'learning_rate': [0.03],
#     'max_depth': [2],
#     'subsample': [0.6],
#     'min_child_weight': [17],
#     'colsample_bytree': [0.8],
#     'gamma': [3, 5],
#     'reg_alpha': [0.1],
#     'reg_lambda': [3]
# }
# ===============================================================
# target+1 best params by fold
# Fold 1: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 2: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 3: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 4: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 5: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 400, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 6: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 0.8}
# Fold 7: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# Fold 8: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 0.3, 'colsample_bytree': 0.8}
# Fold 9: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 400, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 0.3, 'colsample_bytree': 0.8}
# Fold 10: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 400, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 1.0}
# target+3 best params by fold
# Fold 1: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 2: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 3: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 4: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 5: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 5, 'colsample_bytree': 0.8}
# Fold 6: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 7: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 8: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 9: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# Fold 10: {'subsample': 0.6, 'reg_lambda': 3, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 17, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 3, 'colsample_bytree': 0.8}
# ==== target+1 ====
# MAE_1 :  3.4602849496163146
# RMSE_1:  9.358641055672518
# R2 Score_1 :  0.8658999451023801
# Train R2: 0.9739860470940348
# Val R2  : 0.8658999451023801
# Gap     : 0.10808610199165469
# ==== target+3 ====
# MAE_3 :  7.735469741107101
# RMSE_3 :  17.190492617252225
# R2 Score_3 :  0.5867829908012799
# Train R2: 0.6487440505212982
# Val R2  : 0.5867829908012799
# Gap     : 0.06196105972001831
# param_dist1 = {
#     'n_estimators': [200, 300, 400],
#     'learning_rate': [0.05, 0.07],
#     'max_depth': [3],
#     'subsample': [0.8],
#     'min_child_weight': [5, 7, 10],
#     'colsample_bytree': [0.8, 1.0],
#     'gamma': [0.3, 1],
#     'reg_alpha': [0, 0.1],
#     'reg_lambda': [1, 3]
# }

# param_dist3 = {
#     'n_estimators': [200, 300, 700],
#     'learning_rate': [0.03, 0.05],
#     'max_depth': [2],
#     'subsample': [0.6],
#     'min_child_weight': [17],
#     'colsample_bytree': [0.8],
#     'gamma': [3, 5],
#     'reg_alpha': [0.1],
#     'reg_lambda': [1, 3]
# }

In [ ]:
xgbmodel = XGBRegressor(
    objective='reg:squarederror', # 회귀 문제 설정
    random_state=42
)

# 과적합 판단 확인
train_r2_list1=[]
val_r2_list1=[]

train_r2_list3=[]
val_r2_list3=[]

#best params
best_params_1 = []
best_params_3 = []

best_score_1=[]
best_score_3=[]

mae_list_1 = []
rmse_list_1=[]
r2_score_list_1=[]

mae_list_3 = []
rmse_list_3=[]
r2_score_list_3=[]

param_dist1 = {
    'n_estimators': [200, 300, 400],
    'learning_rate': [0.05, 0.07],
    'max_depth': [3],
    'subsample': [0.8],
    'min_child_weight': [5, 7, 10],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0.3, 1],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 3]
}

param_dist3 = {
    'n_estimators': [200, 300, 700],
    'learning_rate': [0.03, 0.05],
    'max_depth': [2],
    'subsample': [0.6],
    'min_child_weight': [17],
    'colsample_bytree': [0.8],
    'gamma': [3, 5],
    'reg_alpha': [0.1],
    'reg_lambda': [1, 3]
}

outer_tscv = TimeSeriesSplit(n_splits=10)
inner_tscv = TimeSeriesSplit(n_splits=3)

for train_idx, val_idx in outer_tscv.split(x_train):
    x_tr,x_val = x_train.iloc[train_idx],x_train.iloc[val_idx]
    y_tr1,y_val1 = y_train_1.iloc[train_idx],y_train_1.iloc[val_idx]
    y_tr3,y_val3 = y_train_3.iloc[train_idx],y_train_3.iloc[val_idx]

    random_search1 = RandomizedSearchCV(
        estimator=xgbmodel,
        param_distributions = param_dist1,
        n_iter = 20,
        scoring = 'neg_mean_squared_error',
        cv = inner_tscv,
        verbose = 1,
        n_jobs=-1,
        random_state=42
    )

    random_search3 = RandomizedSearchCV(
        estimator=xgbmodel,
        param_distributions = param_dist3,
        n_iter = 20,
        scoring = 'neg_mean_squared_error',
        cv = inner_tscv,
        verbose = 1,
        n_jobs=-1,
        random_state=42
    )

    random_search1.fit(x_tr,y_tr1)
    best_model1 = random_search1.best_estimator_

    best_params_1.append(random_search1.best_params_)
    best_score_1.append(random_search1.best_score_)

    pred_tr1=best_model1.predict(x_tr)
    pred_val1=best_model1.predict(x_val)

    train_r2_list1.append(r2_score(y_tr1,pred_tr1))
    val_r2_list1.append(r2_score(y_val1,pred_val1))

    mae1 = mean_absolute_error(y_val1,pred_val1)
    mae_list_1.append(mae1)

    rmse1 = root_mean_squared_error(y_val1,pred_val1)
    rmse_list_1.append(rmse1)

    r2_1 = r2_score(y_val1,pred_val1)
    r2_score_list_1.append(r2_1)

    random_search3.fit(x_tr,y_tr3)
    best_model3 = random_search3.best_estimator_

    best_params_3.append(random_search3.best_params_)
    best_score_3.append(random_search3.best_score_)

    pred_tr3=best_model3.predict(x_tr)
    pred_val3=best_model3.predict(x_val)

    train_r2_list3.append(r2_score(y_tr3,pred_tr3))
    val_r2_list3.append(r2_score(y_val3,pred_val3))

    mae3 = mean_absolute_error(y_val3,pred_val3)
    mae_list_3.append(mae3)

    rmse3 = root_mean_squared_error(y_val3,pred_val3)
    rmse_list_3.append(rmse3)

    r2_3 = r2_score(y_val3,pred_val3)
    r2_score_list_3.append(r2_3)

print("target+1 best params by fold")
for i, p in enumerate(best_params_1, 1):
    print(f"Fold {i}: {p}")

print("target+3 best params by fold")
for i, p in enumerate(best_params_3, 1):
    print(f"Fold {i}: {p}")

print("==== target+1 ====")
print("MAE_1 : ",np.mean(mae_list_1))
print("RMSE_1: ",np.mean(rmse_list_1))
print("R2 Score_1 : ",np.mean(r2_score_list_1))
print("Train R2:", np.mean(train_r2_list1))
print("Val R2  :", np.mean(val_r2_list1))
print("Gap     :", np.mean(train_r2_list1) - np.mean(val_r2_list1))

print("==== target+3 ====")
print("MAE_3 : ",np.mean(mae_list_3))
print("RMSE_3 : ",np.mean(rmse_list_3))
print("R2 Score_3 : ",np.mean(r2_score_list_3))
print("Train R2:", np.mean(train_r2_list3))
print("Val R2  :", np.mean(val_r2_list3))
print("Gap     :", np.mean(train_r2_list3) - np.mean(val_r2_list3))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each

In [ ]:
import gc
gc.collect()

0

빅데이터 분석 과제 설명
제목 성의있게 알아서

In [ ]:
# 데이터 수집

# Apache Spark를 이용하여 데이터 수집. 과제 조건 충족.
# sparksession 초기화(메모리에서 많이 사용. 어떤 정보를 로그인처럼 유지할 때 사용.)
# 매개변수말고 체인으로 실행.
# 예시 코드 갤러리에 있음.
# 파일 이름에는 관심이 없음. 랜딩. 열 컬럼으로 빠르게 획득. 파티션바이
#